# Email Phishing Detection Training Pipeline

This notebook implements a training pipeline for phishing email detection using:
- Text preprocessing (combining subject and body, lowercase)
- Stopword removal 
- TF-IDF vectorization
- Logistic Regression model

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import nltk
from nltk.corpus import stopwords
import pickle

# Download required NLTK data
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/danvaccaro/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
# Load the training dataset
df = pd.read_csv('train_dataset.csv')
print(f"Training dataset shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")
df.head()

Training dataset shape: (34244, 5)
Label distribution:
label
1    17122
0    17122
Name: count, dtype: int64


,sender,subject,body,label,source
0,Joann Braun <rqyegidfkanq@boatbrowser.com>,Re:,IPTV – Die Zukunft des Fernsehens hat begonnen...,1,CEAS_08
1,"""Dintelmann, Peter"" <wstjd.lwgstoezgh@Dresdner...",AW: [perl #46349] Building v5.10.0 64-bit on S...,\n\n> -----Ursprüngliche Nachricht-----\n> Von...,0,CEAS_08
2,Ron Kim <TednearestHansen@circle.net>,Delightsome Bvlgari watches at Replica Classic...,\nIn our online store you can buy replicas of ...,1,CEAS_08
3,Daily Top 10 <Randol-ruberete@fn-dokr.de>,CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,CEAS_08
4,Vicente Erickson <linasssmet@asss.de>,Anonymously and qualitatively gsff bt f,After many years of Rese uf arch and Develo ec...,1,CEAS_08


In [3]:
# Combine subject and body text
def preprocess_text(row):
    subject = str(row['subject']).lower() if pd.notna(row['subject']) else ''
    body = str(row['body']).lower() if pd.notna(row['body']) else ''
    return f"{subject} {body}".strip()

df['combined_text'] = df.apply(preprocess_text, axis=1)

# Get stopwords
stop_words = list(stopwords.words('english'))

In [4]:
# Create the pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=stop_words,
        max_features=10000,  # Limit features to prevent memory issues
        ngram_range=(1, 2),  # Include both unigrams and bigrams
        lowercase=True  # Ensure everything is lowercase
    )),
    ('classifier', LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])

# Train the pipeline on full training set
X = df['combined_text']
y = df['label']
pipeline.fit(X, y)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                                 stop_words=['i', 'me', 'my', 'myself', 'we',
                                             'our', 'ours', 'ourselves', 'you',
                                             "you're", "you've", "you'll",
                                             "you'd", 'your', 'yours',
                                             'yourself', 'yourselves', 'he',
                                             'him', 'his', 'himself', 'she',
                                             "she's", 'her', 'hers', 'herself',
                                             'it', "it's", 'its', 'itself', ...])),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])

In [6]:
# Example prediction function
def predict_email(subject, body):
    # Preprocess the input text
    subject = str(subject).lower() if subject else ''
    body = str(body).lower() if body else ''
    combined = f"{subject} {body}".strip()
    
    # Predict using pipeline
    prediction = pipeline.predict([combined])[0]
    probability = pipeline.predict_proba([combined])[0]
    
    return {
        'prediction': prediction,
        'confidence': probability.max()
    }

# Test with a sample email
sample_subject = "Test subject"
sample_body = "This is a test email body"
result = predict_email(sample_subject, sample_body)
print(f"\nPrediction for sample email:")
print(f"Class: {'Phishing' if result['prediction'] == 1 else 'Legitimate'}")
print(f"Confidence: {result['confidence']:.4f}")


Prediction for sample email:
Class: Legitimate
Confidence: 0.7247


In [7]:
# Save the trained pipeline
print("\nSaving pipeline...")
with open('email_classifier_pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
print("Pipeline saved as 'email_classifier_pipeline.pkl'")

# Example of how to load and use the pipeline
print("\nExample of loading and using saved pipeline:")
print("import pickle")
print("with open('email_classifier_pipeline.pkl', 'rb') as f:")
print("    pipeline = pickle.load(f)")
print("prediction = pipeline.predict(['email text'])[0]")


Saving pipeline...
Pipeline saved as 'email_classifier_pipeline.pkl'

Example of loading and using saved pipeline:
import pickle
with open('email_classifier_pipeline.pkl', 'rb') as f:
    pipeline = pickle.load(f)
prediction = pipeline.predict(['email text'])[0]
